# 🏆 POC 22: Comprehensive Walk-Forward Benchmark: Naive, Optuna-Tuned, Regime-Adaptive & Tri-Horizon Ensembles (1981–2026)

**File**: [`research/notebooks/algo-alpha-execution/22_regime_adaptive_dynamic_horizon_walkforward_backtest.ipynb`](file:///c:/Users/honza/Desktop/projects/stock-analysis/research/notebooks/algo-alpha-execution/22_regime_adaptive_dynamic_horizon_walkforward_backtest.ipynb)  
**Scope**: Definitive multi-decade institutional benchmark across **45.6 Years (1981–2026 / 12,264 daily trading sessions)** comparing **9 distinct strategies**, from default naive baselines to point-in-time Bayesian Optuna-tuned models and multi-horizon confluence ensembles, including **Year-by-Year Annual Profit Curves and Annual Excess Alpha over S&P 500 for every single strategy**.

---

### 🛡️ 9-Strategy Benchmark Lineup:
1. **`1. S&P 500 Index (^GSPC Benchmark)`**: Passive broad-market index.
2. **`2. Point-in-Time Active Universe B&H`**: Equal-weight buy-and-hold of actively trading equities (zero `bfill`).
3. **`3. Naive XGBoost (Default Params, 30d/30d Eq, T+1)`**: Default XGBoost hyperparameters, equal-weight allocation.
4. **`4. Static Purged XGBoost (30d/30d Prop, T+1)`**: Fixed 30-day fundamental drift, forecast-proportional sizing.
5. **`5. Static Purged XGBoost (15d/15d Prop, T+1)`**: Fixed 15-day swing momentum, forecast-proportional sizing.
6. **`6. Optuna-Tuned Purged XGBoost (15d/15d Prop, T+1)`**: Point-in-time Bayesian hyperparameter optimization.
7. **`7. Regime-Adaptive Dynamic Horizon XGBoost`**: Real-time volatility conditioning ($H=5	ext{d}$ in crisis, $H=15	ext{d}$ in moderate vol, $H=35	ext{d}$ in bull).
8. **`8. Tri-Horizon Multi-Model Ensemble (5d/15d/35d)`**: Multi-timeframe confluence blend ($30\% 5	ext{d} + 40\% 15	ext{d} + 30\% 35	ext{d}$).
9. **`9. Optuna-Tuned Tri-Horizon Ensemble (5d/15d/35d)`**: Multi-horizon confluence with Bayesian-tuned hyperparameters across all 3 sub-models.

```
┌────────────────────────────────────────────────────────────────────────────────────────┐
│ COMPREHENSIVE MULTI-DECADE STRATEGY MATRIX (1981–2026)                                 │
│                                                                                        │
│ 1. S&P 500 BENCHMARK         ──► Market Index (^GSPC)                                  │
│ 2. ACTIVE UNIVERSE B&H       ──► Passive Active Basket (Survivorship Baseline)         │
│ 3. NAIVE XGBOOST (Default)   ──► Default Params, Equal Weighting, T+1 Lag              │
│ 4. STATIC PURGED (30d/30d)   ──► Fixed 30d Drift, Prop Conviction Sizing, T+1 Lag      │
│ 5. STATIC PURGED (15d/15d)   ──► Fixed 15d Swing, Prop Conviction Sizing, T+1 Lag      │
│ 6. OPTUNA-TUNED (15d/15d)    ──► Bayesian Hyperparameter Tuning, Prop Sizing, T+1 Lag  │
│ 7. REGIME-ADAPTIVE (Dyn)     ──► Dynamic (H_t, F_t) conditioned on σ_SPY, T+1 Lag      │
│ 8. TRI-HORIZON ENSEMBLE      ──► Multi-Model Confluence (5d/15d/35d), Prop, T+1 Lag    │
│ 9. OPTUNA TRI-HORIZON        ──► 3-Model Confluence + Bayesian Tuning, Prop, T+1 Lag   │
└────────────────────────────────────────────────────────────────────────────────────────┘
```

## 1. Setup & Environment Configuration

In [1]:
import os
import sys
import time
import pandas as pd
import numpy as np
import xgboost as xgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import yfinance as yf
from tqdm.auto import tqdm

# Robust project root discovery
current_dir = os.path.abspath(os.getcwd())
while current_dir and not os.path.exists(os.path.join(current_dir, "src")):
    parent = os.path.dirname(current_dir)
    if parent == current_dir:
        break
    current_dir = parent

PROJECT_ROOT = current_dir
DATA_PATH = os.path.join(PROJECT_ROOT, "data", "processed", "master_panel_1975_2026.parquet")
LOCAL_DATA_DIR = os.path.join(PROJECT_ROOT, "research", "notebooks", "algo-alpha-execution", "data", "fetched")

print(f"📁 Project Root: {PROJECT_ROOT}")
print(f"📁 Loading Half-Century Master Parquet: {DATA_PATH}")

t0 = time.perf_counter()
df_master = pd.read_parquet(DATA_PATH)
df_master['date'] = pd.to_datetime(df_master['date'])

# Precompute forward prediction horizons H = 5d, 15d, 30d, 35d
for h in [5, 15, 30, 35]:
    df_master[f'target_{h}d'] = df_master.groupby('ticker')['close'].transform(lambda s: s.shift(-h) / s - 1.0)

prices_pivot = df_master.pivot(index='date', columns='ticker', values='close').ffill()
daily_rets = prices_pivot.pct_change().fillna(0.0)
daily_rets_mat = daily_rets.values
all_dates = prices_pivot.index
n_days, n_tickers = daily_rets_mat.shape

features = [
    'revenue_growth', 'net_margin', 'sentiment_score', 'rsi_14', 'macd',
    'is_opp_buy', 'is_pol_buy', 'ewma_volatility', 'daily_news_count',
    'daily_news_finbert_sentiment', 'news_volume_intensity',
    'news_decay_tau_1d_ema', 'news_decay_tau_3d_ema', 'news_sentiment_velocity'
]

print(f"✅ Ingested {len(df_master):,} records across {df_master['ticker'].nunique()} tickers ({df_master['date'].min().strftime('%Y-%m-%d')} to {df_master['date'].max().strftime('%Y-%m-%d')}) in {time.perf_counter()-t0:.2f}s!")

📁 Project Root: c:\Users\honza\Desktop\projects\stock-analysis
📁 Loading Half-Century Master Parquet: c:\Users\honza\Desktop\projects\stock-analysis\data\processed\master_panel_1975_2026.parquet


✅ Ingested 638,434 records across 60 tickers (1978-01-03 to 2026-08-27) in 0.69s!


## 2. Ingest S&P 500 Benchmark & Rolling Macro Volatility Signal

In [2]:
start_dt = all_dates[0].strftime('%Y-%m-%d')
end_dt = all_dates[-1].strftime('%Y-%m-%d')

print(f"⏳ Downloading S&P 500 (^GSPC) benchmark data from {start_dt} to {end_dt}...")
spx_raw = yf.download("^GSPC", start=start_dt, end=end_dt, progress=False)
if isinstance(spx_raw.columns, pd.MultiIndex):
    spx_raw.columns = spx_raw.columns.get_level_values(0)

spx_aligned = spx_raw['Close'].reindex(all_dates).ffill().bfill()
spx_equity = (spx_aligned / spx_aligned.iloc[0]) * 100.0
spx_vol_60d = spx_aligned.pct_change().rolling(60).std() * np.sqrt(252.0) * 100.0

print(f"✅ Benchmark data aligned ({len(all_dates)} daily sessions from {start_dt} to {end_dt}).")

⏳ Downloading S&P 500 (^GSPC) benchmark data from 1978-01-03 to 2026-08-27...


✅ Benchmark data aligned (12264 daily sessions from 1978-01-03 to 2026-08-27).


## 3. Strict Purged Walk-Forward Simulation Engine (9 Multi-Decade Strategies)

In [3]:
burnin_end_date = pd.to_datetime('1981-01-02')
eval_start_idx = all_dates.get_loc(burnin_end_date)

w_active_bh   = np.zeros_like(daily_rets_mat)
w_naive_30    = np.zeros_like(daily_rets_mat)
w_static_30   = np.zeros_like(daily_rets_mat)
w_static_15   = np.zeros_like(daily_rets_mat)
w_optuna_15   = np.zeros_like(daily_rets_mat)
w_adaptive    = np.zeros_like(daily_rets_mat)
w_ensemble    = np.zeros_like(daily_rets_mat)
w_opt_ensemble= np.zeros_like(daily_rets_mat)

adaptive_regime_logs = []

# Rebalance Date Lists
F30 = 31
rebal_dates_30 = [d for d in all_dates[::F30] if d >= burnin_end_date]
F15 = 15
rebal_dates_15 = [d for d in all_dates[::F15] if d >= burnin_end_date]

# -----------------------------------------------------------------------------
# 1. 30-DAY CYCLES: ACTIVE B&H, NAIVE XGBOOST (DEFAULT), AND STATIC 30d/30d
# -----------------------------------------------------------------------------
print(f"🚀 Running 30-Day Cycles: Active B&H, Naive XGBoost, and Static 30d/30d ({len(rebal_dates_30)} cycles)...")

for reb_date in tqdm(rebal_dates_30, desc="30d Cycles"):
    t_idx = all_dates.get_loc(reb_date)
    end_idx = min(t_idx + 1 + F30, n_days)
    
    cand_df = df_master[(df_master['date'] == reb_date) & df_master['close'].notnull()]
    if len(cand_df) == 0:
        continue
    
    # 2. Point-in-Time Active Universe B&H
    active_idx = [prices_pivot.columns.get_loc(s) for s in cand_df['ticker'] if s in prices_pivot.columns]
    w_active_bh[t_idx+1:end_idx, active_idx] = 1.0 / len(active_idx)
    
    purge_idx_30 = max(0, t_idx - 30)
    train_30d = df_master[df_master['date'] <= all_dates[purge_idx_30]].tail(100000)
    train_clean = train_30d[train_30d['target_30d'].notnull()]
    
    # 3. Naive XGBoost (Default Params, Equal-Weight Top 50)
    m_naive = xgb.XGBRegressor(n_jobs=-1, random_state=42)
    m_naive.fit(train_clean[features].values, train_clean['target_30d'].values)
    p_naive = pd.Series(m_naive.predict(cand_df[features]), index=cand_df['ticker']).nlargest(min(50, len(cand_df)))
    top_idx_naive = [prices_pivot.columns.get_loc(s) for s in p_naive.index if s in prices_pivot.columns]
    w_naive_30[t_idx+1:end_idx, top_idx_naive] = 1.0 / len(top_idx_naive)
    
    # 4. Static Purged XGBoost (30d/30d Prop)
    m_30 = xgb.XGBRegressor(n_estimators=40, max_depth=4, learning_rate=0.05, n_jobs=-1, random_state=42, tree_method='hist')
    m_30.fit(train_clean[features].values, train_clean['target_30d'].values)
    p_30 = pd.Series(m_30.predict(cand_df[features]), index=cand_df['ticker']).nlargest(min(50, len(cand_df)))
    sc_30 = p_30.clip(lower=0.0001)
    w_prop_30 = (sc_30 / sc_30.sum()).values
    top_idx_30 = [prices_pivot.columns.get_loc(s) for s in p_30.index if s in prices_pivot.columns]
    w_static_30[t_idx+1:end_idx, top_idx_30] = w_prop_30[:len(top_idx_30)]

# -----------------------------------------------------------------------------
# 2. 15-DAY CYCLES: STATIC 15d, OPTUNA 15d, TRI-HORIZON & OPTUNA TRI-HORIZON
# -----------------------------------------------------------------------------
print(f"🚀 Running 15-Day Cycles: Static 15d, Optuna 15d, Tri-Horizon, and Optuna Tri-Horizon ({len(rebal_dates_15)} cycles)...")

# Cache for annual Optuna parameters
optuna_params_15d = {'n_estimators': 40, 'max_depth': 4, 'learning_rate': 0.05, 'subsample': 0.85, 'colsample_bytree': 0.85}
optuna_params_3m = {
    5: {'n_estimators': 35, 'max_depth': 3, 'learning_rate': 0.05},
    15: {'n_estimators': 40, 'max_depth': 4, 'learning_rate': 0.05},
    35: {'n_estimators': 40, 'max_depth': 4, 'learning_rate': 0.05}
}
last_optuna_year = None

for reb_date in tqdm(rebal_dates_15, desc="15d Cycles"):
    t_idx = all_dates.get_loc(reb_date)
    end_idx = min(t_idx + 1 + F15, n_days)
    curr_year = reb_date.year
    
    cand_df = df_master[(df_master['date'] == reb_date) & df_master['close'].notnull()]
    if len(cand_df) == 0:
        continue
        
    purge_idx_15 = max(0, t_idx - 15)
    train_15d = df_master[df_master['date'] <= all_dates[purge_idx_15]].tail(100000)
    train_clean_15 = train_15d[train_15d['target_15d'].notnull()]
    
    # 5. Static Purged XGBoost (15d/15d Prop)
    m_15 = xgb.XGBRegressor(n_estimators=40, max_depth=4, learning_rate=0.05, n_jobs=-1, random_state=42, tree_method='hist')
    m_15.fit(train_clean_15[features].values, train_clean_15['target_15d'].values)
    p_15 = pd.Series(m_15.predict(cand_df[features]), index=cand_df['ticker']).nlargest(min(50, len(cand_df)))
    sc_15 = p_15.clip(lower=0.0001)
    w_prop_15 = (sc_15 / sc_15.sum()).values
    top_idx_15 = [prices_pivot.columns.get_loc(s) for s in p_15.index if s in prices_pivot.columns]
    w_static_15[t_idx+1:end_idx, top_idx_15] = w_prop_15[:len(top_idx_15)]
    
    # Periodic Annual Optuna Hyperparameter Optimization
    if curr_year != last_optuna_year:
        last_optuna_year = curr_year
        # Run fast 8-trial study on recent purged slice
        X_tune = train_clean_15[features].tail(30000).values
        y_tune = train_clean_15['target_15d'].tail(30000).values
        def obj_15(trial):
            p_dict = {
                'n_estimators': trial.suggest_int('n_estimators', 25, 55),
                'max_depth': trial.suggest_int('max_depth', 3, 5),
                'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.08, log=True),
                'subsample': trial.suggest_float('subsample', 0.7, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
                'tree_method': 'hist', 'n_jobs': -1, 'random_state': 42
            }
            split = int(len(X_tune) * 0.7)
            mt = xgb.XGBRegressor(**p_dict)
            mt.fit(X_tune[:split], y_tune[:split])
            return np.mean((mt.predict(X_tune[split:]) - y_tune[split:]) ** 2)
            
        study_15 = optuna.create_study(direction='minimize')
        study_15.optimize(obj_15, n_trials=6)
        optuna_params_15d = {**study_15.best_params, 'tree_method': 'hist', 'n_jobs': -1, 'random_state': 42}
        
    # 6. Optuna-Tuned Purged XGBoost (15d/15d Prop)
    m_opt_15 = xgb.XGBRegressor(**optuna_params_15d)
    m_opt_15.fit(train_clean_15[features].values, train_clean_15['target_15d'].values)
    p_opt_15 = pd.Series(m_opt_15.predict(cand_df[features]), index=cand_df['ticker']).nlargest(min(50, len(cand_df)))
    sc_opt_15 = p_opt_15.clip(lower=0.0001)
    w_prop_opt_15 = (sc_opt_15 / sc_opt_15.sum()).values
    top_idx_opt_15 = [prices_pivot.columns.get_loc(s) for s in p_opt_15.index if s in prices_pivot.columns]
    w_optuna_15[t_idx+1:end_idx, top_idx_opt_15] = w_prop_opt_15[:len(top_idx_opt_15)]
    
    # 8. Tri-Horizon Multi-Model Ensemble (5d + 15d + 35d Confluence)
    purge_5 = max(0, t_idx - 5)
    train_5 = df_master[df_master['date'] <= all_dates[purge_5]].tail(100000)
    m_5 = xgb.XGBRegressor(n_estimators=30, max_depth=3, learning_rate=0.05, n_jobs=-1, random_state=42, tree_method='hist')
    m_5.fit(train_5[features].values, train_5['target_5d'].values)
    
    purge_35 = max(0, t_idx - 35)
    train_35 = df_master[df_master['date'] <= all_dates[purge_35]].tail(100000)
    m_35 = xgb.XGBRegressor(n_estimators=30, max_depth=3, learning_rate=0.05, n_jobs=-1, random_state=42, tree_method='hist')
    m_35.fit(train_35[features].values, train_35['target_35d'].values)
    
    p5 = (m_5.predict(cand_df[features]) - np.mean(m_5.predict(cand_df[features]))) / (np.std(m_5.predict(cand_df[features])) + 1e-5)
    p15 = (m_15.predict(cand_df[features]) - np.mean(m_15.predict(cand_df[features]))) / (np.std(m_15.predict(cand_df[features])) + 1e-5)
    p35 = (m_35.predict(cand_df[features]) - np.mean(m_35.predict(cand_df[features]))) / (np.std(m_35.predict(cand_df[features])) + 1e-5)
    
    p_blend = pd.Series(0.30 * p5 + 0.40 * p15 + 0.30 * p35, index=cand_df['ticker']).nlargest(min(50, len(cand_df)))
    sc = p_blend - p_blend.min() + 0.0001
    w_prop = (sc / sc.sum()).values
    top_idx = [prices_pivot.columns.get_loc(s) for s in p_blend.index if s in prices_pivot.columns]
    w_ensemble[t_idx+1:end_idx, top_idx] = w_prop[:len(top_idx)]
    
    # 9. Optuna-Tuned Tri-Horizon Ensemble
    p15_opt = (m_opt_15.predict(cand_df[features]) - np.mean(m_opt_15.predict(cand_df[features]))) / (np.std(m_opt_15.predict(cand_df[features])) + 1e-5)
    p_blend_opt = pd.Series(0.30 * p5 + 0.40 * p15_opt + 0.30 * p35, index=cand_df['ticker']).nlargest(min(50, len(cand_df)))
    sc_opt_ens = p_blend_opt - p_blend_opt.min() + 0.0001
    w_prop_opt_ens = (sc_opt_ens / sc_opt_ens.sum()).values
    top_idx_opt_ens = [prices_pivot.columns.get_loc(s) for s in p_blend_opt.index if s in prices_pivot.columns]
    w_opt_ensemble[t_idx+1:end_idx, top_idx_opt_ens] = w_prop_opt_ens[:len(top_idx_opt_ens)]

# -----------------------------------------------------------------------------
# 3. REGIME-ADAPTIVE DYNAMIC HORIZON WALK-FORWARD (Adaptive H_t, F_t)
# -----------------------------------------------------------------------------
print(f"🚀 Running Regime-Adaptive Dynamic Horizon Walk-Forward...")
curr_t = eval_start_idx

while curr_t < n_days - 1:
    reb_date = all_dates[curr_t]
    vol_val = spx_vol_60d.iloc[curr_t]
    
    if vol_val > 22.0:
        H_dyn, F_dyn = 5, 5
        regime_label = "Crisis Shock (High Vol)"
    elif vol_val > 14.0:
        H_dyn, F_dyn = 15, 15
        regime_label = "Moderate Swing Momentum"
    else:
        H_dyn, F_dyn = 35, 30
        regime_label = "Low-Vol Fundamental Drift"
        
    end_idx = min(curr_t + 1 + F_dyn, n_days)
    cand_df = df_master[(df_master['date'] == reb_date) & df_master['close'].notnull()]
    if len(cand_df) == 0:
        curr_t += F_dyn
        continue
        
    purge_idx = max(0, curr_t - H_dyn)
    train_df = df_master[df_master['date'] <= all_dates[purge_idx]].tail(100000)
    train_clean = train_df[train_df[f'target_{H_dyn}d'].notnull()]
    
    m_dyn = xgb.XGBRegressor(n_estimators=40, max_depth=4, learning_rate=0.05, n_jobs=-1, random_state=42, tree_method='hist')
    m_dyn.fit(train_clean[features].values, train_clean[f'target_{H_dyn}d'].values)
    
    p = pd.Series(m_dyn.predict(cand_df[features]), index=cand_df['ticker']).nlargest(min(50, len(cand_df)))
    sc = p.clip(lower=0.0001)
    w_prop = (sc / sc.sum()).values
    top_idx = [prices_pivot.columns.get_loc(s) for s in p.index if s in prices_pivot.columns]
    w_adaptive[curr_t+1:end_idx, top_idx] = w_prop[:len(top_idx)]
    
    adaptive_regime_logs.append({
        'Date': reb_date, 'S&P 500 60d Vol (%)': round(vol_val, 2),
        'Selected Horizon H*': f"{H_dyn}d", 'Selected Cadence F*': f"{F_dyn}d",
        'Regime': regime_label
    })
    curr_t += F_dyn

print(f"✅ All 9 Walk-Forward Simulations Completed Successfully!")

🚀 Running 30-Day Cycles: Active B&H, Naive XGBoost, and Static 30d/30d (371 cycles)...


30d Cycles:   0%|          | 0/371 [00:00<?, ?it/s]

🚀 Running 15-Day Cycles: Static 15d, Optuna 15d, Tri-Horizon, and Optuna Tri-Horizon (767 cycles)...


15d Cycles:   0%|          | 0/767 [00:00<?, ?it/s]

🚀 Running Regime-Adaptive Dynamic Horizon Walk-Forward...


✅ All 9 Walk-Forward Simulations Completed Successfully!


## 4. Multi-Decade Performance Analytics & Alpha Matrix (1981–2026)

In [4]:
eval_dates = all_dates[all_dates >= burnin_end_date]

curves = {
    '1. S&P 500 Index (^GSPC Benchmark)': (spx_aligned.loc[eval_dates] / spx_aligned.loc[eval_dates].iloc[0]) * 100.0,
    '2. Point-in-Time Active Universe B&H (No Lookahead)': pd.Series(np.cumprod(1.0 + np.sum(daily_rets_mat[eval_start_idx:] * w_active_bh[eval_start_idx:], axis=1)) * 100.0, index=eval_dates),
    '3. Naive XGBoost (Default Params, 30d/30d Eq, T+1)': pd.Series(np.cumprod(1.0 + np.sum(daily_rets_mat[eval_start_idx:] * w_naive_30[eval_start_idx:], axis=1)) * 100.0, index=eval_dates),
    '4. Static Purged XGBoost (30d Rebal, 30d Fwd, Prop, T+1)': pd.Series(np.cumprod(1.0 + np.sum(daily_rets_mat[eval_start_idx:] * w_static_30[eval_start_idx:], axis=1)) * 100.0, index=eval_dates),
    '5. Static Purged XGBoost (15d Rebal, 15d Fwd, Prop, T+1)': pd.Series(np.cumprod(1.0 + np.sum(daily_rets_mat[eval_start_idx:] * w_static_15[eval_start_idx:], axis=1)) * 100.0, index=eval_dates),
    '6. Optuna-Tuned Purged XGBoost (15d/15d Prop, T+1)': pd.Series(np.cumprod(1.0 + np.sum(daily_rets_mat[eval_start_idx:] * w_optuna_15[eval_start_idx:], axis=1)) * 100.0, index=eval_dates),
    '7. Regime-Adaptive Dynamic Horizon XGBoost (Prop, T+1)': pd.Series(np.cumprod(1.0 + np.sum(daily_rets_mat[eval_start_idx:] * w_adaptive[eval_start_idx:], axis=1)) * 100.0, index=eval_dates),
    '8. Tri-Horizon Multi-Model Ensemble (5d/15d/35d Blend, T+1)': pd.Series(np.cumprod(1.0 + np.sum(daily_rets_mat[eval_start_idx:] * w_ensemble[eval_start_idx:], axis=1)) * 100.0, index=eval_dates),
    '9. Optuna-Tuned Tri-Horizon Ensemble (3-Model Bayesian, T+1)': pd.Series(np.cumprod(1.0 + np.sum(daily_rets_mat[eval_start_idx:] * w_opt_ensemble[eval_start_idx:], axis=1)) * 100.0, index=eval_dates)
}

df_master_curves = pd.DataFrame(curves, index=eval_dates).reset_index().rename(columns={'index': 'date'})

def compute_analytics(series, spx_series, rf=0.03):
    r_strat = series.pct_change().dropna()
    r_spx = spx_series.pct_change().dropna()
    aligned = pd.concat([r_strat, r_spx], axis=1).dropna()
    r_strat, r_spx = aligned.iloc[:, 0], aligned.iloc[:, 1]
    
    n_years = len(r_strat) / 252.0
    total_ret = (series.iloc[-1] / series.iloc[0]) - 1.0
    cagr = (series.iloc[-1] / series.iloc[0]) ** (1.0 / max(n_years, 0.1)) - 1.0
    ann_excess = (r_strat.mean() * 252.0) - rf
    ann_vol = r_strat.std() * np.sqrt(252.0)
    sharpe = ann_excess / ann_vol if ann_vol > 0 else 0.0
    downside_vol = r_strat[r_strat < 0].std() * np.sqrt(252.0)
    sortino = ann_excess / downside_vol if downside_vol > 0 else 0.0
    
    drawdown = (series - series.cummax()) / series.cummax()
    max_dd = drawdown.min()
    calmar = cagr / abs(max_dd) if abs(max_dd) > 0 else 0.0
    
    cov_matrix = np.cov(r_strat, r_spx)
    beta = cov_matrix[0, 1] / cov_matrix[1, 1] if cov_matrix[1, 1] > 0 else 1.0
    spx_cagr = (spx_series.iloc[-1] / spx_series.iloc[0]) ** (1.0 / max(n_years, 0.1)) - 1.0
    alpha = (cagr - rf) - beta * (spx_cagr - rf)
    
    return {
        'Total Return (%)': total_ret * 100.0,
        'CAGR (%)': cagr * 100.0,
        'Sharpe Ratio': sharpe,
        'Sortino Ratio': sortino,
        'Max Drawdown (%)': max_dd * 100.0,
        'Calmar Ratio': calmar,
        'Market Beta (β)': beta,
        'Jensen Alpha (α %)': alpha * 100.0
    }

analytics_records = []
for name, s in curves.items():
    analytics_records.append({'Strategy / Model': name, **compute_analytics(s, curves['1. S&P 500 Index (^GSPC Benchmark)'])})

df_performance_table = pd.DataFrame(analytics_records)
print("=== 9-STRATEGY PERFORMANCE & RISK MATRIX (1981–2026 / 45.6 YEARS) ===")
df_performance_table

=== 9-STRATEGY PERFORMANCE & RISK MATRIX (1981–2026 / 45.6 YEARS) ===


,Strategy / Model,Total Return (%),CAGR (%),Sharpe Ratio,Sortino Ratio,Max Drawdown (%),Calmar Ratio,Market Beta (β),Jensen Alpha (α %)
0,1. S&P 500 Index (^GSPC Benchmark),5.529823e+03,9.230016,0.416663,0.523635,-56.775388,0.162571,1.000000,0.000000
1,2. Point-in-Time Active Universe B&H (No Looka...,2.997233e+05,19.166886,0.908929,1.166969,-45.710596,0.419309,0.962668,10.169452
2,"3. Naive XGBoost (Default Params, 30d/30d Eq, ...",3.013078e+05,19.180645,0.913503,1.175747,-41.250790,0.464976,0.955007,10.230938
3,"4. Static Purged XGBoost (30d Rebal, 30d Fwd, ...",8.592282e+05,21.947228,0.949775,1.235391,-41.150268,0.533344,1.044514,12.439890
4,"5. Static Purged XGBoost (15d Rebal, 15d Fwd, ...",9.341810e+05,22.170805,0.940472,1.223979,-47.055399,0.471164,1.064216,12.540724
5,"6. Optuna-Tuned Purged XGBoost (15d/15d Prop, ...",6.032647e+05,21.006307,0.919718,1.190216,-47.207435,0.444979,1.040315,11.525130
6,7. Regime-Adaptive Dynamic Horizon XGBoost (Pr...,1.242847e+06,22.937088,0.917978,1.188389,-51.498450,0.445394,1.093343,13.125541
7,8. Tri-Horizon Multi-Model Ensemble (5d/15d/35...,2.560012e+06,24.898260,0.958589,1.280560,-41.864442,0.594735,1.135447,14.824405
8,9. Optuna-Tuned Tri-Horizon Ensemble (3-Model ...,2.734948e+06,25.079216,0.965163,1.293939,-40.475809,0.619610,1.134637,15.010410


## 5. Cumulative Equity Curves (Log Scale: 1981–2026) & Underwater Drawdowns

In [5]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=('<b>Half-Century Walk-Forward Equity Curves (Log Scale: 1981–2026)</b>',
                                    '<b>Underwater Drawdown Curves (%)</b>'))

palette = [
    '#636EFA',  # 1. SP500
    '#FFA15A',  # 2. Active Universe B&H
    '#7F7F7F',  # 3. Naive Default
    '#AB63FA',  # 4. Static 30d/30d
    '#FFDF00',  # 5. Static 15d/15d
    '#19D3F3',  # 6. Optuna 15d
    '#00CC96',  # 7. Regime-Adaptive
    '#EF553B',  # 8. Tri-Horizon Ensemble
    '#FF6692'   # 9. Optuna Tri-Horizon Ensemble (Hero)
]

for idx, (name, s) in enumerate(curves.items()):
    c = palette[idx % len(palette)]
    is_hero = ('8.' in name or '9.' in name or '6.' in name)
    fig.add_trace(go.Scatter(
        x=df_master_curves['date'], y=s, name=name,
        line=dict(color=c, width=3.2 if is_hero else 1.8)
    ), row=1, col=1)
    
    dd = ((s - s.cummax()) / s.cummax()) * 100.0
    fig.add_trace(go.Scatter(
        x=df_master_curves['date'], y=dd, name=f"{name} DD", showlegend=False,
        line=dict(color=c, width=1.5)
    ), row=2, col=1)

fig.update_yaxes(type="log", row=1, col=1, title="<b>Portfolio Value ($ Log Scale)</b>")
fig.update_yaxes(row=2, col=1, title="<b>Drawdown (%)</b>")

fig.update_layout(
    template='plotly_dark', width=1200, height=850,
    title='<b>9-Strategy Institutional Benchmark (1981–2026): Naive vs Optuna vs Regime-Adaptive vs Tri-Horizon</b>',
    margin=dict(l=60, r=320, t=80, b=60),
    legend=dict(orientation='v', yanchor='top', y=1.0, xanchor='left', x=1.02, title=dict(text='<b>Strategy / Model</b>'))
)
fig.show()

## 6. Year-by-Year Annual Profit Curves & Annual Excess Alpha over S&P 500 (All 9 Strategies)

In [6]:
# Compute Annual Return Matrix for each strategy
annual_ret_records = []
years_list = sorted(list(set(df_master_curves['date'].dt.year)))

for yr in years_list:
    yr_df = df_master_curves[df_master_curves['date'].dt.year == yr]
    if len(yr_df) < 2:
        continue
    row = {'Year': yr}
    for name in curves.keys():
        ret_val = (yr_df[name].iloc[-1] / yr_df[name].iloc[0] - 1.0) * 100.0
        row[name] = round(ret_val, 2)
    annual_ret_records.append(row)

df_annual_returns = pd.DataFrame(annual_ret_records)

# Compute Annual Excess Alpha over S&P 500 for ALL strategies
df_annual_alpha = pd.DataFrame({'Year': df_annual_returns['Year']})
for name in curves.keys():
    if 'S&P 500' not in name:
        clean_col = name.split('.')[1].strip().split('(')[0].strip() + " Alpha (%)"
        df_annual_alpha[clean_col] = round(df_annual_returns[name] - df_annual_returns['1. S&P 500 Index (^GSPC Benchmark)'], 2)

# Plot 1: Annual Profit Curves (Line + Marker Chart across 45 Years)
fig_annual_curves = go.Figure()

for idx, name in enumerate(curves.keys()):
    c = palette[idx % len(palette)]
    is_hero = ('8.' in name or '9.' in name or '6.' in name)
    fig_annual_curves.add_trace(go.Scatter(
        x=df_annual_returns['Year'], y=df_annual_returns[name],
        mode='lines+markers', name=name.split('.')[1].strip().split('(')[0].strip(),
        line=dict(color=c, width=3.2 if is_hero else 1.8),
        marker=dict(size=7 if is_hero else 5)
    ))

fig_annual_curves.add_hline(y=0.0, line=dict(color='white', width=1, dash='dash'))

fig_annual_curves.update_layout(
    template='plotly_dark', width=1200, height=600,
    title='<b>Year-by-Year Annual Profit Curves (1981–2026 / 45 Years)</b><br><sup>Annual returns (%) across all 9 strategies</sup>',
    xaxis=dict(title='<b>Year</b>', dtick=2),
    yaxis=dict(title='<b>Annual Total Return (%)</b>'),
    margin=dict(l=60, r=60, t=90, b=60),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5)
)
fig_annual_curves.show()

# Plot 2: Comparative Annual Excess Alpha Curves over S&P 500 (All Strategies)
fig_alpha_curves = go.Figure()

for idx, col in enumerate([c for c in df_annual_alpha.columns if c != 'Year']):
    c = palette[(idx + 1) % len(palette)]
    is_hero = ('Tri-Horizon' in col or 'Optuna' in col)
    fig_alpha_curves.add_trace(go.Scatter(
        x=df_annual_alpha['Year'], y=df_annual_alpha[col],
        mode='lines+markers', name=col,
        line=dict(color=c, width=3.0 if is_hero else 1.6),
        marker=dict(size=6 if is_hero else 4)
    ))

fig_alpha_curves.add_hline(y=0.0, line=dict(color='white', width=1.5, dash='dash'))

fig_alpha_curves.update_layout(
    template='plotly_dark', width=1200, height=580,
    title='<b>Comparative Annual Excess Alpha Generated over S&P 500 (% / Year)</b><br><sup>Shows how much each strategy beat or lagged the market index in every individual year</sup>',
    xaxis=dict(title='<b>Year</b>', dtick=2),
    yaxis=dict(title='<b>Excess Return (%) vs S&P 500</b>'),
    margin=dict(l=60, r=60, t=90, b=60),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5)
)
fig_alpha_curves.show()

# Plot 3: Multi-Panel Grouped Bar Chart of Excess Alpha
fig_alpha_bars = make_subplots(
    rows=4, cols=2,
    subplot_titles=[c for c in df_annual_alpha.columns if c != 'Year'],
    horizontal_spacing=0.08, vertical_spacing=0.10
)

for idx, col in enumerate([c for c in df_annual_alpha.columns if c != 'Year']):
    r = (idx // 2) + 1
    c_idx = (idx % 2) + 1
    c_color = palette[(idx + 1) % len(palette)]
    
    fig_alpha_bars.add_trace(go.Bar(
        x=df_annual_alpha['Year'], y=df_annual_alpha[col],
        name=col, marker=dict(color=c_color), showlegend=False
    ), row=r, col=c_idx)
    
    fig_alpha_bars.add_hline(y=0.0, line=dict(color='white', width=1, dash='dash'), row=r, col=c_idx)
    fig_alpha_bars.update_xaxes(title_text="Year", dtick=4, row=r, col=c_idx)
    fig_alpha_bars.update_yaxes(title_text="Alpha %", row=r, col=c_idx)

fig_alpha_bars.update_layout(
    template='plotly_dark', width=1200, height=1050,
    title='<b>Individual Strategy Annual Excess Alpha Bars over S&P 500 (1981–2026)</b><br><sup>Positive bars indicate years where the strategy generated pure excess alpha over the market</sup>',
    margin=dict(l=50, r=50, t=100, b=50)
)
fig_alpha_bars.show()

print("=== ANNUAL EXCESS ALPHA MATRIX OVER S&P 500 (%) ===")
df_annual_alpha

=== ANNUAL EXCESS ALPHA MATRIX OVER S&P 500 (%) ===


,Year,Point-in-Time Active Universe B&H Alpha (%),Naive XGBoost Alpha (%),Static Purged XGBoost Alpha (%),Optuna-Tuned Purged XGBoost Alpha (%),Regime-Adaptive Dynamic Horizon XGBoost Alpha (%),Tri-Horizon Multi-Model Ensemble Alpha (%),Optuna-Tuned Tri-Horizon Ensemble Alpha (%)
0,1981,12.95,12.95,9.73,10.63,9.19,9.84,10.46
1,1982,26.28,26.28,56.33,44.48,48.18,55.87,57.19
2,1983,5.65,5.65,2.95,4.48,12.67,6.65,10.21
3,1984,6.04,6.04,4.29,4.96,2.02,2.64,2.39
4,1985,9.63,9.63,17.54,15.11,15.84,19.65,19.40
5,1986,9.19,9.19,11.11,11.29,9.31,11.36,11.82
6,1987,14.61,14.61,19.73,17.36,18.32,23.40,22.90
7,1988,7.99,7.99,8.62,8.23,7.80,5.28,6.22
8,1989,17.36,17.36,19.70,19.67,23.36,22.11,21.65
9,1990,16.07,16.07,13.47,15.36,13.50,8.91,8.85


## 7. Decade-by-Decade Quantitative Breakdown (1981–2026)

In [7]:
decades = [
    ('1980s (1981–1989)', pd.to_datetime('1981-01-02'), pd.to_datetime('1989-12-29')),
    ('1990s (1990–1999)', pd.to_datetime('1990-01-02'), pd.to_datetime('1999-12-31')),
    ('2000s (2000–2009)', pd.to_datetime('2000-01-03'), pd.to_datetime('2009-12-31')),
    ('2010s (2010–2019)', pd.to_datetime('2010-01-04'), pd.to_datetime('2019-12-31')),
    ('2020s (2020–2026)', pd.to_datetime('2020-01-02'), pd.to_datetime('2026-08-27'))
]

decade_records = []
for dec_name, d_start, d_end in decades:
    row = {'Decade / Market Era': dec_name}
    for name, s in curves.items():
        s_sub = s[(s.index >= d_start) & (s.index <= d_end)]
        if len(s_sub) > 0:
            sub_ret = (s_sub.iloc[-1] / s_sub.iloc[0] - 1.0) * 100.0
            row[name.split('.')[1].strip().split('(')[0].strip()] = round(sub_ret, 2)
    decade_records.append(row)

df_decades = pd.DataFrame(decade_records)
print("=== DECADE-BY-DECADE TOTAL RETURN BREAKDOWN (%) ===")
df_decades

=== DECADE-BY-DECADE TOTAL RETURN BREAKDOWN (%) ===


,Decade / Market Era,S&P 500 Index,Point-in-Time Active Universe B&H,Naive XGBoost,Static Purged XGBoost,Optuna-Tuned Purged XGBoost,Regime-Adaptive Dynamic Horizon XGBoost,Tri-Horizon Multi-Model Ensemble,Optuna-Tuned Tri-Horizon Ensemble
0,1980s (1981–1989),159.20,557.57,557.57,739.63,682.39,723.14,779.70,821.53
1,1990s (1990–1999),308.48,1305.76,1198.49,1791.85,1629.92,1694.23,2664.26,2533.64
2,2000s (2000–2009),-23.37,153.53,179.61,271.04,211.32,449.76,372.78,380.42
3,2010s (2010–2019),185.16,341.40,329.05,353.74,336.83,339.45,447.45,482.44
4,2020s (2020–2026),135.61,180.49,185.16,232.63,212.93,231.91,281.51,277.85


## 8. Regime-Switching Timeline: How the Strategy Adapted $(H_t, F_t)$ Over Time

In [8]:
df_regime_logs = pd.DataFrame(adaptive_regime_logs)

fig_regimes = px.scatter(
    df_regime_logs, x='Date', y='S&P 500 60d Vol (%)',
    color='Regime', size=[10]*len(df_regime_logs),
    title="<b>Dynamic Horizon Switching Events across 45.6 Years (1981–2026)</b>",
    color_discrete_map={
        'Crisis Shock (High Vol)': '#EF553B',
        'Moderate Swing Momentum': '#FFDF00',
        'Low-Vol Fundamental Drift': '#00CC96'
    }
)

fig_regimes.update_layout(template='plotly_dark', width=1100, height=480)
fig_regimes.show()

## 9. Export All 9 Strategy Simulations & Annual Alpha Matrices to Excel

In [9]:
out_path = os.path.join(LOCAL_DATA_DIR, "comprehensive_9_strategy_walkforward_benchmark_poc.xlsx")
with pd.ExcelWriter(out_path) as writer:
    df_master_curves.to_excel(writer, sheet_name='daily_equity_curves', index=False)
    df_performance_table.to_excel(writer, sheet_name='performance_summary', index=False)
    df_annual_returns.to_excel(writer, sheet_name='annual_returns_matrix', index=False)
    df_annual_alpha.to_excel(writer, sheet_name='annual_excess_alpha_matrix', index=False)
    df_decades.to_excel(writer, sheet_name='decade_breakdown', index=False)
    df_regime_logs.to_excel(writer, sheet_name='regime_switching_logs', index=False)

print(f"💾 Successfully exported Comprehensive 9-Strategy Benchmark & Annual Alpha Matrices to: {out_path}")

💾 Successfully exported Comprehensive 9-Strategy Benchmark & Annual Alpha Matrices to: c:\Users\honza\Desktop\projects\stock-analysis\research\notebooks\algo-alpha-execution\data\fetched\comprehensive_9_strategy_walkforward_benchmark_poc.xlsx
